# 💎 Chapter 14: Singular Value Decomposition (SVD)

---

## 14.1 Introduction: The Crown Jewel of Linear Algebra

In Chapter 13, we learned that Eigendecomposition ($A = V \Lambda V^{-1}$) breaks a matrix down into its core rotational axes and scaling factors. However, Eigendecomposition has strict limitations: it only applies to square matrices, and fails completely on defective matrices.

In Data Science, our datasets (Design Matrices) are almost always tall, rectangular matrices (e.g., 10,000 patients, 50 features). How do we decompose a rectangular matrix?

Enter **Singular Value Decomposition (SVD)**. SVD is a generalization of eigendecomposition that is mathematically guaranteed to work on **any matrix in the universe**, regardless of its shape, contents, or rank. 

SVD breaks any matrix $\mathbf{A}$ into three matrices:
$$ \mathbf{A} = \mathbf{U} \mathbf{\Sigma} \mathbf{V}^T $$

- $\mathbf{U}$ (Left Singular Vectors): An orthogonal matrix capturing the row space (relationship between observations).
- $\mathbf{\Sigma}$ (Singular Values): A diagonal matrix containing positive scalars, sorted in descending order of importance.
- $\mathbf{V}^T$ (Right Singular Vectors): An orthogonal matrix capturing the column space (relationship between features).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set plotting style
%matplotlib inline
np.random.seed(42)
print("Scientific libraries successfully imported for Chapter 14.")

## 14.2 Computing SVD in Python

Let's take a rectangular matrix (e.g., $4 \times 3$) and decompose it. 

In practice, we usually compute the "Economy" or "Reduced" SVD (`full_matrices=False` in NumPy). Instead of padding matrices with useless zeros to make them perfectly square, the Economy SVD computes only the components strictly necessary to reconstruct the original matrix, saving massive amounts of memory.

In [ ]:
# Create a tall rectangular matrix (4 observations, 3 features)
A = np.random.randint(1, 10, size=(4, 3))

# Perform Economy SVD
# U will be 4x3, S will be a 1D array of 3 values, Vt will be 3x3
U, S, Vt = np.linalg.svd(A, full_matrices=False)

print("Original Matrix A (4x3):\n", A)
print("\nLeft Singular Vectors U (4x3):\n", np.round(U, 3))
print("\nSingular Values S (1D Array):\n", np.round(S, 3))
print("\nRight Singular Vectors V^T (3x3):\n", np.round(Vt, 3))

# Reconstruct the matrix: U @ Sigma @ V^T
# Note: np.linalg.svd returns S as a 1D array to save memory, 
# so we must convert it into a diagonal matrix using np.diag(S)
Sigma = np.diag(S)
A_reconstructed = U @ Sigma @ Vt

print("\nReconstructed Matrix (U @ Sigma @ V^T):\n", np.round(A_reconstructed))
print("Does the reconstruction match A?", np.allclose(A, A_reconstructed))

## 14.3 The Secret Link: SVD and Eigendecomposition

If SVD and Eigendecomposition are related, what exactly is the mathematical link?

If we take our non-square data matrix $\mathbf{A}$ and multiply it by its own transpose to create a square, symmetric covariance matrix ($\mathbf{A}^T\mathbf{A}$), an amazing thing happens:
1. The **Right Singular Vectors ($\mathbf{V}$)** of $\mathbf{A}$ are exactly the **Eigenvectors** of $\mathbf{A}^T\mathbf{A}$.
2. The **Singular Values ($\Sigma$)** of $\mathbf{A}$ are exactly the **Square Roots of the Eigenvalues** of $\mathbf{A}^T\mathbf{A}$.

This is why SVD is the preferred algorithm for doing PCA (Principal Component Analysis). Instead of doing the computationally risky $\mathbf{A}^T\mathbf{A}$ multiplication, we just run SVD directly on the raw data matrix $\mathbf{A}$ to extract the principal components ($\mathbf{V}$).

In [ ]:
# 1. Create a Covariance-style matrix: A^T A
AtA = A.T @ A

# 2. Perform Eigendecomposition on A^T A
eigenvalues, eigenvectors = np.linalg.eigh(AtA)

# Sort eigenvalues and eigenvectors in descending order (to match SVD convention)
sorted_indices = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[sorted_indices]
eigenvectors = eigenvectors[:, sorted_indices]

print("--- Comparing Eigendecomposition of A^T A with SVD of A ---\n")

# Compare Vectors (Note: Eigenvectors might have flipped signs, which is mathematically valid)
print("Eigenvectors of A^T A:\n", np.round(eigenvectors, 3))
print("\nRight Singular Vectors (V) from SVD (Notice they are the same columns!):\n", np.round(Vt.T, 3))

# Compare Values
print("\nSquare Root of Eigenvalues of A^T A:", np.round(np.sqrt(np.abs(eigenvalues)), 3))
print("Singular Values from SVD of A        :", np.round(S, 3))

## 14.4 Low-Rank Matrix Approximation (Data Compression)

The most famous application of SVD is **Data Compression / Dimensionality Reduction**. 

The singular values in $\Sigma$ are strictly sorted from largest (most important) to smallest (least important). The Eckart-Young-Mirsky Theorem proves that if we throw away the smallest singular values (by setting them to zero) and rebuild the matrix, we get the mathematically optimal approximation of our original dataset using less memory.

This is called a **Low-Rank Approximation**. Let's visualize this by "compressing" a 2D image (since an image is just a matrix of pixels).

In [ ]:
from skimage import data
from skimage.color import rgb2gray

# 1. Load a sample image and convert it to grayscale (a 2D matrix)
# We will use an astronaut image provided by skimage
img = rgb2gray(data.astronaut())
print(f"Original Image Matrix Shape: {img.shape}")

# 2. Perform SVD on the image matrix
U_img, S_img, Vt_img = np.linalg.svd(img, full_matrices=False)

# 3. Create a function to reconstruct the image using only the top 'k' singular values
def compress_image(U, S, Vt, k):
    # Take only the first k columns of U, top k singular values, and first k rows of Vt
    U_k = U[:, :k]
    Sigma_k = np.diag(S[:k])
    Vt_k = Vt[:k, :]
    # Reconstruct
    return U_k @ Sigma_k @ Vt_k

# 4. Reconstruct using different ranks
k_values = [5, 20, 100]

plt.figure(figsize=(16, 4))

# Plot Original
plt.subplot(1, 4, 1)
plt.imshow(img, cmap='gray')
plt.title(f"Original ({img.shape[0]}x{img.shape[1]})")
plt.axis('off')

# Plot Approximations
for i, k in enumerate(k_values):
    img_compressed = compress_image(U_img, S_img, Vt_img, k)
    plt.subplot(1, 4, i+2)
    plt.imshow(img_compressed, cmap='gray')
    plt.title(f"Rank k={k} Approximation")
    plt.axis('off')

plt.tight_layout()
plt.show()

print("Notice how at k=5, we only capture the most basic shapes (highest variance). ")
print("By k=100, the image is virtually indistinguishable from the original, even though we threw away over 80% of the data!")

## 14.5 Chapter Summary

In Chapter 14, we explored the ultimate matrix factorization:
- **Universality:** SVD works on any matrix (tall, wide, square, singular, or full rank).
- **The Formula:** $\mathbf{A} = \mathbf{U} \mathbf{\Sigma} \mathbf{V}^T$, breaking data into left singular vectors (row patterns), singular values (importance weights), and right singular vectors (feature patterns).
- **Connection to Covariance:** SVD is a direct, numerically safer shortcut to finding the eigenvectors of $\mathbf{A}^T\mathbf{A}$.
- **Data Compression:** Because $\Sigma$ is sorted by importance, dropping the smallest singular values yields optimal low-rank approximations. This is the exact math running behind JPEG image compression, background noise reduction, and recommendation engines like Netflix.